In [ ]:
from datasets import load_from_disk

In [ ]:
dataset = load_from_disk(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/mmb0-1b_smb1-1b_1p/mmb0-1b_smb1-1b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset')

In [ ]:
dataset

In [ ]:
item = dataset[0]

In [ ]:
item

In [ ]:
import math
distances = [math.sqrt(dx**2 + dy**2) for dx, dy in zip(list(item['rel_x_coord']), list(item['rel_y_coord']))]
print(distances)

In [ ]:
sample_size = 0.5

In [ ]:
#dataset = load_from_disk(f'/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_{sample_size}_sample_32_None_None_None_None_None_shifted_log_knn_10.dataset')
#dataset = load_from_disk(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_6.dataset')
dataset = load_from_disk(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhs1000-39b_1p/xhs1000-39b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset')

In [ ]:
item = dataset[0]

In [ ]:
item

In [ ]:
len(item['gene_tokens'])

In [ ]:
import pickle

#split_path = f'/nfs/team361/sb75/DATASETS/tokenizer/hst_corpus_110m_{sample_size}_sample_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_full.pkl'
split_path = f'/nfs/team361/sb75/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_6_precomputed_split_train.pkl'

with open(split_path, 'rb') as f:
    split = pickle.load(f)

print(split[:100])
print(split[-100:])
print(len(split))

In [ ]:
import pickle

#nz_tokens_path = f'/nfs/team361/sb75/DATASETS/tokenizer/hst_corpus_110m_{sample_size}_sample_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_full.pkl'
nz_tokens_path = f'/nfs/team361/sb75/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_6_n_nonzero_tokens_train.pkl'

with open(nz_tokens_path, 'rb') as f:
    nz_tokens = pickle.load(f)

print(nz_tokens[:100])
print(nz_tokens[-100:])

In [ ]:
train_dataset = dataset.select(split)

ds_nz_tokens = train_dataset['n_nonzero_tokens']

print(ds_nz_tokens[:100])
print(ds_nz_tokens[-100:])

In [ ]:
import pickle

idx_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx_split_train.pkl'

with open(idx_path, 'rb') as f:
    idx = pickle.load(f)

In [ ]:
len(idx)

In [ ]:
from datasets import load_from_disk

In [ ]:
ds = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_ordered_full.dataset')

In [ ]:
ds

In [ ]:
from pathlib import Path

root = Path("/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer")
pattern = "None_None_None_None_None_shifted_log_knn_6_7168.dataset"

# Recursively search for directories containing the pattern in their name
matching_dirs = [
    str(p) for p in root.rglob("*")
    if p.is_dir() and pattern in p.name
]

print(f"Found {len(matching_dirs)} matching directories:")
for d in matching_dirs:
    #if d == '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhb1002-AT12-1b_1p/xhb1002-AT12-1b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset':
    #    continue
    #elif d == '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1015-R6-28b_1p/xhl1015-R6-28b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset':
    #    continue
    #elif d == '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xho1012-Hrv162-2b_1p/xho1012-Hrv162-2b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset':
    #    continue
    #else:
    ds = load_from_disk(d)
    if 'batch_token' not in ds.features.keys():
        print(d)
        print(ds)
        print('===')

In [ ]:
/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhb1002-AT12-1b_1p/xhb1002-AT12-1b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import time
import timeit
import random

import torch
from datasets import load_from_disk

from nichejepa.masks.block_masking  import BlockMaskCollator
from nichejepa.datasets.cell_datasets import init_cell_dataset
from nichejepa.datasets.dataloaders import init_dataloader_and_sampler, CustomDistributedLengthGroupedSampler

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


ImportError: cannot import name 'init_cell_dataset' from 'nichejepa.datasets.cell_datasets' (/nfs/team361/sb75/nichejepa/src/nichejepa/datasets/cell_datasets.py)

In [ ]:
batch_size = 124

In [4]:
# Load Huggingface Dataset
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
#data_path = '/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset' # 88626282
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset' # 88363649
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10.dataset' # 112578039
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_old.dataset' # 112578039
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_6.dataset'
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhs1000-39b_1p/xhs1000-39b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset'
dataset = load_from_disk(data_path)
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 112578039
})

In [5]:
batch_values = list(dataset['batch_value'])

In [17]:
cell_ids = list(dataset['cell_id'])


KeyboardInterrupt



In [68]:
for i in range(1, 10):
    ds = load_from_disk(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1020-{i}-16b_1p/xhl1020-{i}-16b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset')
    try:
        print(ds['cell_id'])
        print(ds['batch_value'])
    except:
        print(ds['cell_id'])
        print(i)

Column(['1020_batch3_0', '1020_batch3_1', '1020_batch3_2', '1020_batch3_3', '1020_batch3_4'])
Column([[801], [801], [801], [801], [801]])
Column(['1020_batch6_0', '1020_batch6_1', '1020_batch6_2', '1020_batch6_3', '1020_batch6_4'])
Column([[804], [804], [804], [804], [804]])
Column(['1020_batch9_0', '1020_batch9_1', '1020_batch9_2', '1020_batch9_3', '1020_batch9_4'])
Column(['1020_batch9_0', '1020_batch9_1', '1020_batch9_2', '1020_batch9_3', '1020_batch9_4'])
3
Column(['1020_batch14_0', '1020_batch14_1', '1020_batch14_2', '1020_batch14_3', '1020_batch14_4'])
Column([[812], [812], [812], [812], [812]])
Column(['1020_batch15_0', '1020_batch15_1', '1020_batch15_2', '1020_batch15_3', '1020_batch15_4'])
Column([[813], [813], [813], [813], [813]])
Column(['1020_batch11_0', '1020_batch11_1', '1020_batch11_2', '1020_batch11_3', '1020_batch11_4'])
Column([[809], [809], [809], [809], [809]])
Column(['1020_batch13_0', '1020_batch13_1', '1020_batch13_2', '1020_batch13_3', '1020_batch13_4'])
Column

FileNotFoundError: Directory /lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1020-9-16b_1p/xhl1020-9-16b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset not found

In [61]:
ds = load_from_disk(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1021-1-2b_1p/xhl1021-1-2b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset')
# print(ds['batch_value'])
print(len(ds))

968236


In [62]:
item = ds[0]

'1021_batch0'

In [23]:
import scanpy as sc
adata = sc.read_h5ad('/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhl1020-1-16b_1p/output-XETG00055__0043482__TB16-1337_LLL_C-S5__20250331__114617.h5ad')

In [10]:
batch_vals = [val_list[0] if val_list else None for val_list in batch_values]

In [12]:
idx = [i for i, x in enumerate(batch_vals) if x is None]

In [56]:
i = 0
while True:
    print(dataset[idx[i]]['cell_id'])
    i += 30000

1020_batch9_0
1020_batch9_30000
1020_batch9_60000
1020_batch9_90000
1020_batch9_120000
1020_batch9_150000
1020_batch9_180000
1020_batch9_210000
1020_batch9_240000
1020_batch9_270000
1020_batch9_300000
1020_batch9_330000
1020_batch9_360000
1020_batch9_390000
1020_batch9_420000
1020_batch9_450000
1020_batch9_480000
1020_batch9_510000
1020_batch9_540000
1020_batch9_570000
1020_batch9_600000
1020_batch9_630000
1020_batch9_660000
1020_batch9_690000
1020_batch9_720000
1020_batch9_750000
1020_batch9_780000
1020_batch9_810000
1020_batch9_840000
1020_batch9_870000
1020_batch9_900000
1020_batch9_930000
1020_batch9_960000
1020_batch9_990000
1020_batch9_1020000
1020_batch9_1050000
1020_batch9_1080000
1020_batch9_1110000
1020_batch5_25138
1020_batch5_55138
1020_batch5_85138
1020_batch5_115138
1020_batch5_145138
1020_batch5_175138
1020_batch5_205138
1020_batch5_235138
1020_batch5_265138
1020_batch5_295138
1020_batch5_325138
1020_batch5_355138
1020_batch5_385138
1020_batch5_415138
1020_batch5_445138


IndexError: list index out of range

In [40]:
len(idx)

5396478

In [36]:
cell_ids = list(dataset['cell_id'])


KeyboardInterrupt



In [16]:
len(idx)

5396478

In [14]:
min(idx)

100527119

In [15]:
max(idx)

110950510

In [6]:
max(batch_values)

TypeError: '>' not supported between instances of 'NoneType' and 'list'

In [5]:
cell_ids = list(dataset['cell_id'])

In [6]:
from collections import Counter

counter = Counter(cell_ids)
duplicates = [item for item, count in counter.items() if count > 1]

print(duplicates)

[autoreload of nichejepa.masks.block_masking failed: Traceback (most recent call last):
  File "/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", line 883, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/nfs/team361/sb75/nichejepa/src/nichejepa/masks/block_masking.py", line 17, in <module>
    from ..masks.utils import create_controlled_mask_context_target, configure_attention_masks
ImportError: cannot im

In [ ]:
print(duplicates[0])

In [ ]:
start_idx = []
for i in range(40):
    idx = cell_ids.index(f'1010_batch{i}_0')
    print(idx)
    start_idx.append(idx)

In [ ]:
min(start_idx)

In [ ]:
cell_ids.index('1010_batch1_0')

In [ ]:
cell_ids.index('1010_batch2_0')

In [ ]:
cell_ids.index('1010_batch7_0')

In [ ]:
cell_ids.index('101_batch7_0')

In [ ]:
dataset[len(dataset)]

In [ ]:
dataset[70853333]['cell_id']

In [ ]:
len(cell_ids)

In [ ]:
from datasets import concatenate_datasets

start = 70438096
end = 70853334

ds1 = dataset.select(range(0, start))
ds2 = dataset.select(range(end, len(dataset)))

ds_new = concatenate_datasets([ds1, ds2])


In [ ]:
ds_new # 112578038

In [ ]:
ds_new.save_to_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_new.dataset')

In [ ]:
70488912 - 70891421

In [ ]:
cell_ids = list(ds_new['cell_id'])

In [ ]:
from collections import Counter

counter = Counter(cell_ids)
duplicates = [item for item, count in counter.items() if count > 1]

print(duplicates)

In [ ]:
len(ds_new)

In [ ]:
len(cell_ids)

In [ ]:
cell_ids_set = set(cell_ids)
print(len(cell_ids_set))

In [ ]:
cell_ids_set = set(cell_ids)
print(len(cell_ids_set))

In [ ]:
cell_ids.index('1010_batch0_1')

In [ ]:
cell_ids.index('1011_batch0_1')

In [ ]:
dataset # 112578039   112539951

In [ ]:
# 112578039

In [ ]:
item = dataset[88363649]

In [ ]:
item['cell_id']

In [ ]:
# 6 neighbors
for i in range(0, len(dataset), 10000):
    item = dataset[i]
    if item['gene_tokens'][6145] == 0:
        print(i)

In [ ]:
cell_ids = dataset[88363650]['cell_id']

In [ ]:
dataset = dataset.select(range(0, 88363649))

In [ ]:
# 10 neighbors
for i in range(88000000, len(dataset), 10000):
    item = dataset[i]
    if item['gene_tokens'][2561] == 0:
        print(i)

In [ ]:
item = dataset[0]

In [ ]:
item['gene_tokens'][2561]

In [ ]:
##### import math
xs = item['rel_x_coord']
ys = item['rel_y_coord']
distances = [math.sqrt(dx**2 + dy**2) for dx, dy in zip(list(xs), list(ys))]
print(distances)

In [ ]:
# (Optional) Filter Dataset
dataset_id = '1000'

dataset = dataset.filter(
    lambda ids: [i.split('_')[0] == dataset_id for i in ids],
    input_columns="cell_id",
    batched=True,
    batch_size=10000,
    num_proc=6,
    desc="Filtering by dataset_id"
)
dataset

In [ ]:
datasets_folder_path = '/nfs/team361/sb75/DATASETS' # '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = f'{datasets_folder_path}/tokenizer/token_dictionary_homo_sapiens.pkl'

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
max_cls_tokens = sum(1 for key in token_dict if "cls" in key)
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

In [ ]:
token_dict['spt_batch']

def flatten_special_tokens(batch):
    return {
        "batch_token": [x[0] for x in batch["batch_token"]]
        "gene_panel_token": [x[0] for x in batch["gene_panel_token"]]
        "assay_token": [x[0] for x in batch["assay_token"]]
        "species_token": [x[0] for x in batch["species_token"]]
        "tissue_token": [x[0] for x in batch["tissue_token"]]
    }

dataset = dataset.map(
    flatten_special_tokens,
    batched=True,
    num_proc=6)

In [ ]:
cell_ids = list(filtered_dataset['cell_id'])

In [ ]:
set([cell_id.split('_')[1] for cell_id in cell_ids])

In [ ]:
item = dataset[0]
print(item.keys())

In [ ]:
n_nz_tokens_cell = 135
print(item['gene_tokens'][:n_nz_tokens_cell+1])
print(item['gene_expr'][:n_nz_tokens_cell+1])
print(item['rel_x_coord'][:n_nz_tokens_cell+1])
print(item['rel_y_coord'][:n_nz_tokens_cell+1])
print(item['n_nonzero_tokens'])
print(item['cell_id'])

In [ ]:
n_nz_tokens_cell = 257
print(item['gene_tokens'][:n_nz_tokens_cell+1])
print(item['gene_expr'][:n_nz_tokens_cell+1])
print(item['rel_x_coord'][:n_nz_tokens_cell+1])
print(item['rel_y_coord'][:n_nz_tokens_cell+1])
print(item['n_nonzero_tokens'])
print(item['cell_id'])

In [ ]:
n_nz_tokens_cell = 513
print(item['gene_tokens'][:n_nz_tokens_cell+1])
print(item['gene_expr'][:n_nz_tokens_cell+1])
print(item['rel_x_coord'][:n_nz_tokens_cell+1])
print(item['rel_y_coord'][:n_nz_tokens_cell+1])
print(item['n_nonzero_tokens'])
print(item['cell_id'])

In [ ]:
print(item['gene_tokens'][2304:])
print(item['gene_expr'][2304:])
print(item['rel_x_coord'][2304:])
print(item['rel_y_coord'][2304:])
print(item['n_nonzero_tokens'])
print(item['cell_id'])

In [ ]:
import math
distances = [math.sqrt(dx**2 + dy**2) for dx, dy in zip(list(item['rel_x_coord']), list(item['rel_y_coord']))]
print(set(distances))

In [ ]:
print(token_dict_reverse[14043])
print(token_dict_reverse[11475])
#print(token_dict_reverse[12472])
#print(token_dict_reverse[18728])
#print(token_dict_reverse[14821])
#print(token_dict_reverse[587])
print(token_dict_reverse[18775])
print(token_dict_reverse[14043])
print(token_dict_reverse[12472])
print(token_dict_reverse[18728])
print(token_dict_reverse[19557])
print(token_dict_reverse[18775])

In [ ]:
import scanpy as sc
#adata = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhs1000-39b_1p/adata_batch3.h5ad')
adata = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhb23-1b_1p/Xenium_V1_Human_Brain_GBM_FFPE_outs.h5ad')

In [ ]:
import squidpy as sq
sq.gr.spatial_neighbors(adata,
                        coord_type='generic',
                        spatial_key='spatial',
                        n_neighs=10,
                        set_diag=False,
                        ) 
knn_connectivities = adata.obsp['spatial_connectivities'] 
knn_distances = adata.obsp['spatial_distances']

In [ ]:
connectivities_idx = np.nonzero(knn_connectivities[0].toarray())
connectivities_idx

In [ ]:
knn_distances[0,    list([int(element) for element in connectivities_idx[1]])].toarray()

In [ ]:
i = 0
neighbors_i = adata.obsp[
                'spatial_connectivities'][i].nonzero()[1]
print(neighbors_i)
cell_start = adata.obsp['spatial_distances'].indptr[i]
cell_end = adata.obsp['spatial_distances'].indptr[i+1]
cell_distances = adata.obsp[
    'spatial_distances'].data[cell_start:cell_end]
print(cell_start)
print(cell_end)
print(cell_distances)
sorted_indices = np.argsort(cell_distances)
print(sorted_indices)
for j, k in enumerate(neighbors_i[sorted_indices]):
    print(j)
    print(k)

In [ ]:
# Get sorted indices of neighbor cells based on (lower) distance to
# index cell
cell_start = adata_neigh.obsp['spatial_distances'].indptr[i]
cell_end = adata_neigh.obsp['spatial_distances'].indptr[i+1]
cell_distances = adata_neigh.obsp[
    'spatial_distances'].data[cell_start:cell_end]

sorted_indices = np.argsort(cell_distances)
assert len(neighbors_i) == len(cell_distances), (
    'Number of neighbors does not equal number of distances.')

# Loop through distance-sorted neighbor cells and add gene,
# segment tokens, counts, and relative coords
for j, k in enumerate(neighbors_i[sorted_indices]):
    adata_dict['gene_tokens_neighborhood'][i] = np.hstack(
        (adata_dict['gene_tokens_neighborhood'][i],
         adata_dict['gene_tokens_cell_neigh'][k]))
    adata_dict['gene_expr_neighborhood'][i] = np.hstack(
        (adata_dict['gene_expr_neighborhood'][i],
         adata_dict['gene_expr_cell_neigh'][k]))
    adata_dict['seg_tokens_neighborhood'][i] = np.hstack(
        (adata_dict['seg_tokens_neighborhood'][i],
         [j + 1] * len(adata_dict['gene_tokens_cell_neigh'][k])))

In [ ]:
ENSG00000177666
ENSG00000165269
ENSG00000254772
ENSG00000177666
ENSG00000169439
ENSG00000253729

In [ ]:
import numpy as np
idx1 = np.where((adata.var['ensembl_id'] == 'ENSG00000177666').values)[0]
print(idx1)
idx2 = np.where((adata.var['ensembl_id'] == 'ENSG00000165269').values)[0]
print(idx2)
#idx3 = np.where((adata.var['ensembl_id'] == 'ENSG00000169439').values)[0]
#print(idx3)
#idx4 = np.where((adata.var['ensembl_id'] == 'ENSG00000253729').values)[0]
#print(idx4)
#idx5 = np.where((adata.var['ensembl_id'] == 'ENSG00000182718').values)[0]
#print(idx5)
#idx6 = np.where((adata.var['ensembl_id'] == 'ENSG00000030110').values)[0]
#print(idx6)
idx3 = np.where((adata.var['ensembl_id'] == 'ENSG00000254772').values)[0]
print(idx3)
idx4 = np.where((adata.var['ensembl_id'] == 'ENSG00000177666').values)[0]
print(idx4)
idx5 = np.where((adata.var['ensembl_id'] == 'ENSG00000169439').values)[0]
print(idx5)
idx6 = np.where((adata.var['ensembl_id'] == 'ENSG00000253729').values)[0]
print(idx6)

idx7 = np.where((adata.var['ensembl_id'] == 'ENSG00000276409').values)[0]
print(idx7)
idx8 = np.where((adata.var['ensembl_id'] == 'ENSG00000254772').values)[0]
print(idx8)

In [ ]:
print(np.log(adata.X[0].toarray()[:, idx1]+1))
print(np.log(adata.X[0].toarray()[:, idx2]+1))
print(np.log(adata.X[2217].toarray()[:, idx3]+1))
print(np.log(adata.X[2217].toarray()[:, idx4]+1))
print(np.log(adata.X[2222].toarray()[:, idx5]+1))
print(np.log(adata.X[2222].toarray()[:, idx6]+1))
print(np.log(adata.X[2225].toarray()[:, idx7]+1))
print(np.log(adata.X[2225].toarray()[:, idx8]+1))

In [ ]:
columns = list(dataset.features.keys())
columns.remove("cell_id")
dataset.set_format(
    type=None, #"torch",
    columns=columns,
    output_all_columns=True)
dataset

In [ ]:
# Define params for Cell Dataset
token_dict_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl'

with open(token_dict_folder_path, 'rb') as file:
    token_dict = pickle.load(file)
vocab_size = len(token_dict)

seq_len_cell = 256
seq_len_neighborhood = 2560
tokenizer_type = 'cell_graph'
gt_type = 'combined'
cell_pos_enc = 'coord'
special_tokens = []
sampling_strategy = None
sep_gene_tokens_neb = False

In [ ]:
#with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl', "rb") as f: 
#with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl', "rb") as f: 
#    n_nonzero_tokens = pickle.load(f)
n_nonzero_tokens = None

In [ ]:
#with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', "rb") as f: 
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', "rb") as f: 
    train_idx = pickle.load(f)

In [ ]:
#train_dataset = dataset.select(train_idx)
train_dataset = dataset
train_dataset

In [ ]:
#n_nonzero_tokens_correct = list(train_dataset['n_nonzero_tokens'])

In [ ]:
n_nonzero_tokens[:100]

In [ ]:
columns = list(train_dataset.features.keys())
columns.remove("cell_id")
train_dataset.set_format(
    type="torch", # "torch",
    columns=columns,
    output_all_columns=True)

In [ ]:
def run():
    for i in range(batch_size):
        data = train_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.070871 seconds
# Average per run: 0.620088 seconds

In [ ]:
random_range = random.sample(range(0, 81361647), batch_size)

def run():
    for i in random_range:
        data = train_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 3.405504 seconds
# Average per run: 4.063928 seconds

In [ ]:
def run():
    data = train_dataset[list(range(batch_size))]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.197345 seconds

In [ ]:
random_range = random.sample(range(0, 81361647), batch_size)

def run():
    data = train_dataset[random_range]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 5.533628 seconds

In [ ]:
# Initialize Cell Dataset
cell_dataset = init_cell_dataset(
    dataset=train_dataset,
    vocab_size=vocab_size,
    seq_len_cell=seq_len_cell,
    seq_len_neighborhood=seq_len_neighborhood,
    tokenizer_type=tokenizer_type,
    gt_type=gt_type,
    cell_pos_enc=cell_pos_enc,
    special_tokens=special_tokens,
    sampling_strategy=sampling_strategy,
    n_nonzero_tokens_list=n_nonzero_tokens,
    include_cell_id=False,
    sep_gene_tokens_neb=sep_gene_tokens_neb)

In [ ]:
len(cell_dataset)

In [ ]:
cell_data = cell_dataset[0]

In [ ]:
import torch
torch.set_printoptions(threshold=float('inf'))

In [ ]:
cell_data.keys()

In [ ]:
n_nz_tokens_cell = 135
print(cell_data['tokens'][:n_nz_tokens_cell+1])
print(cell_data['values'][:n_nz_tokens_cell+1])
print(cell_data['rel_x_coords'][:n_nz_tokens_cell+1])
print(cell_data['rel_y_coords'][:n_nz_tokens_cell+1])
print(cell_data['positions'][:n_nz_tokens_cell+1])
print(cell_data['segments'][:n_nz_tokens_cell+1])

In [ ]:
n_tokens_per_cell = 256
print(cell_data['tokens'][:n_tokens_per_cell+2])
print(cell_data['values'][:n_tokens_per_cell+2])
print(cell_data['rel_x_coords'][:n_tokens_per_cell+2])
print(cell_data['rel_y_coords'][:n_tokens_per_cell+2])
print(cell_data['positions'][:n_tokens_per_cell+2])
print(cell_data['segments'][:n_tokens_per_cell+2])

In [ ]:
len(cell_data['tokens'])

In [ ]:
batch_size = 144

In [ ]:
import timeit

def run():
    for i in range(batch_size):
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.468819 seconds

In [ ]:
random_range = random.sample(range(0, 81361647), batch_size)

import timeit

def run():
    for i in random_range:
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 4.059347 seconds

In [ ]:
cell_data['segments'][:100]

In [ ]:
cell_data['positions'][:513]

In [ ]:
cell_data['rel_x_coords'][:513]

In [ ]:
cell_data['segments'][:513]

In [ ]:
random_range_restricted = sorted(random.sample(range(0, 10000000), batch_size))

In [ ]:
import timeit

def run():
    for i in random_range_restricted:
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds") # 0,421 seconds

In [ ]:
random_range_restricted_2 = sorted(random.sample(range(0, 10000), batch_size))

In [ ]:
import timeit

def run():
    for i in random_range_restricted_2:
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds") # 0,421 seconds

In [ ]:
cell_dataset[1000]['rel_x_coords'][-500:]

In [ ]:
cell_dataset[1000]['rel_y_coords'][-500:]

In [ ]:
import timeit

def run():
    for i in range(batch_size):
        cell_data = cell_dataset[81361646 - i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.468819 seconds

In [ ]:
n_targets = 1
n_contexts = 1
n_segments = 11
n_special_tokens = 0
per_block_mask_ratio = 0.6
sample_segments = True

In [ ]:
mask_collator = BlockMaskCollator(
    n_targets=n_targets,
    n_contexts=n_contexts,
    n_segments=n_segments,
    seq_len_cell=seq_len_cell,
    seq_len_neighborhood=seq_len_neighborhood,
    n_special_tokens=n_special_tokens,
    per_block_mask_ratio=per_block_mask_ratio,
    sample_segments=sample_segments,
    sample_gene_masks=True)

In [ ]:
world_size = 1
rank = 0
pin_memory = True
num_workers = 0

In [ ]:
distributed = True # True, False

if not distributed:
    loader, sampler = init_dataloader_and_sampler(
        cell_dataset=cell_dataset,
        batch_size=batch_size,
        distributed=distributed,
        world_size=world_size,
        rank=rank,
        collate_fn=mask_collator,
        pin_memory=pin_memory,
        num_workers=num_workers,
        drop_last=True,
        #prefetch_factor=4,
        persistent_workers=False)
else:
    loader, sampler = init_dataloader_and_sampler(
        cell_dataset=cell_dataset,
        batch_size=batch_size,
        distributed=distributed,
        world_size=world_size,
        rank=rank,
        collate_fn=mask_collator,
        pin_memory=pin_memory,
        num_workers=num_workers,
        drop_last=True,
        #prefetch_factor=4,
        persistent_workers=False)

In [ ]:
#device = 'cuda'
device = 'cpu'

In [ ]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
    if itr == 0:
        start = time.time()
    elapsed = time.time() - start
    print(f"Time: {elapsed:.6f} seconds")
    if itr == 3:
        break

In [ ]:
print("Iter with 32 GPUS:", int(len(loader) / 32 / 16))

In [ ]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
    if itr == 0:
        start = time.time()
    elapsed = time.time() - start
    print(f"Time: {elapsed:.6f} seconds")
    if itr == 3:
        break

In [ ]:
import timeit
import torch, time

def run_one_iter():
    for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
        for key, val in udata.items():
            udata[key] = val.to(device, non_blocking=True)
        masks_enc = [u.to(device, non_blocking=True) for u in masks_enc]
        masks_pred = [u.to(device, non_blocking=True) for u in masks_pred]
        masks_attention = masks_attention.to(device, non_blocking=True)
        break
    
    if device == "cuda":
        torch.cuda.synchronize()

# --- First iteration timing ---
start = time.time()
run_one_iter()
if device == "cuda":
    torch.cuda.synchronize()
first_elapsed = time.time() - start
print(f"First iteration: {first_elapsed:.6f} seconds")

# --- Average over next 10 iterations ---
elapsed = timeit.timeit(run_one_iter, number=10)
print(f"Average over next 10: {elapsed / 10:.6f} seconds")

In [ ]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
    for key, val in udata.items():
        udata[key] = val.to(device, non_blocking=True)
    masks_enc = [u.to(device, non_blocking=True) for u in masks_enc]
    masks_pred = [u.to(device, non_blocking=True) for u in masks_pred]
    masks_attention = masks_attention.to(device, non_blocking=True)
    break

In [ ]:
udata

In [ ]:
(udata['tokens'][0] != 0).sum()

In [ ]:
(udata['values'][0] != 0).sum()

In [ ]:
(udata['rel_x_coords'][0] != 0).sum()

In [ ]:
import math
distances = [math.sqrt(dx**2 + dy**2) for dx, dy in zip(udata['rel_x_coords'][0], udata['rel_y_coords'][0])]

In [ ]:
udata['segments'][0][513:1800]

In [ ]:
udata['rel_x_coords'][0][:513]

In [ ]:
distances

In [ ]:
(udata['positions'][0] != 0).sum()

In [ ]:
(udata['segments'][0] != 0).sum()

In [ ]:
(udata['tokens'][1] != 0).sum()

In [ ]:
(udata['tokens'][2] != 0).sum()

In [ ]:
(udata['tokens'][30] != 0).sum()

In [ ]:
udata['tokens'].shape

In [ ]:
masks_attention[2].sum()

In [ ]:
idx_temp = [79009928, 61334070, 61012720, 59885839, 51564208, 51071983, 50578755, 51490262, 21762874, 50991337, 20596439, 49076277, 55395062, 50791923, 51597330]

In [ ]:
dataset_tmp = train_dataset.select(idx_temp)

In [ ]:
len(sampler.lengths)

In [ ]:
dataset_tmp['n_nonzero_tokens']

In [ ]:
g = torch.Generator()
g.manual_seed(0)
indices = sampler._get_length_grouped_indices(generator=g)

In [ ]:
indices[:100]

In [ ]:
train_dataset

In [ ]:
ordered_dataset = train_dataset.select(indices[-100:])

In [ ]:
train_dataset[79009928]['n_nonzero_tokens']

In [ ]:
list(ordered_dataset['n_nonzero_tokens'])

In [ ]:
print(torch.count_nonzero(cell_dataset[79009928]['tokens']))
print(torch.count_nonzero(cell_dataset[61334070]['tokens']))
print(torch.count_nonzero(cell_dataset[61012720]['tokens']))
print(torch.count_nonzero(cell_dataset[59885839]['tokens']))
print(torch.count_nonzero(cell_dataset[51564208]['tokens']))
print(torch.count_nonzero(cell_dataset[51071983]['tokens']))
print(torch.count_nonzero(cell_dataset[50578755]['tokens']))
print(torch.count_nonzero(cell_dataset[51490262]['tokens']))

In [ ]:
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "wb") as f: 
    pickle.dump(indices, f)

In [ ]:
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices2 = pickle.load(f)

In [ ]:
indices[:100]

In [ ]:
indices2[:100]

In [ ]:
test = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2])

In [ ]:
mask = test == 1

In [ ]:
mask

In [ ]:
torch.nonzero(mask, as_tuple=True)[0]

In [ ]:
import numpy as np
import scanpy as sc

from nichejepa.preprocessors.graph import construct_neighbor_graph

In [ ]:
adata_neigh = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhe1008-1-20b_1p/section020_SOB26.h5ad')

In [ ]:
n_neighs = 10
radius = None
delaunay = None

In [ ]:
print('Computing spatial neighborhood...')
# Construct neighbor graph
adata_neigh = construct_neighbor_graph(
    adata_neigh,
    n_neighs=n_neighs,
    radius=radius,
    delaunay=delaunay,
    include_self_loop=False,
    compute_neighbor_counts=False)

In [ ]:
adata_neigh.obsp['spatial_connectivities'][0].indices

In [ ]:
adata_neigh.obsp['spatial_connectivities'][0].nonzero()[1]

In [ ]:
adata_neigh.obsp['spatial_distances'].indptr[2]

In [ ]:
adata_neigh.obsp['spatial_distances'].indptr[3]

In [ ]:
cell_start = adata_neigh.obsp['spatial_distances'].indptr[0]
cell_end = adata_neigh.obsp['spatial_distances'].indptr[0+1]
cell_distances = adata_neigh.obsp[
    'spatial_distances'].data[cell_start:cell_end]
cell_distances

In [ ]:
rel_x_coord = []
rel_y_coord = []

neighbor_indices = adata_neigh.obsp[
    'spatial_connectivities'][0].indices
for j in neighbor_indices:
    rel_x_coord.extend([(
        adata_neigh.obsm['spatial'][j, 0] - adata_neigh.obsm['spatial'][0, 0]
        ).tolist()])
    rel_y_coord.extend([(
        adata_neigh.obsm['spatial'][j, 1] - adata_neigh.obsm['spatial'][0, 1]
        ).tolist()])

print(rel_x_coord)
print(rel_y_coord)

In [ ]:
-7-(-10)

In [ ]:
np.sqrt(5.62371826171875**2 + 10.005859375 **2)

In [ ]:
np.sqrt(rel_x_coord**2 + rel_y_coord**2)

In [ ]:
adata_neigh